# Hybrid Service Blame Analysis

Replication and extension of **Chen et al. (2025)** — *Agentic AI as A Scapegoat* (SSRN-6032194).

This notebook demonstrates a modern Python pipeline for:
1. Detecting service failures and human-agent blame toward AI in chat sessions
2. Measuring customer emotion, engagement, and purchase outcomes
3. Estimating causal effects using a two-stage control-function design

> **Note:** The original paper uses proprietary Taobao chat logs. This notebook uses transparent synthetic data with the same variable structure so you can run everything without private data access.

## 1. Setup

In [ ]:
!pip -q install pandas numpy scikit-learn statsmodels matplotlib seaborn plotly

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from sklearn.metrics import accuracy_score, roc_auc_score

sns.set_theme(style="whitegrid")
print("Libraries loaded.")

## 2. Generate Session Data (Paper-Aligned Structure)

We create session-level records with the same core variables used in the paper:
- service failure sessions only
- blame indicator
- customer emotion, log engagement, purchase
- peer blame instrument and controls

In [ ]:
BLAME_PHRASES = [
    "the system made a mistake",
    "our ai misunderstood",
    "this issue was caused by the bot",
    "the chatbot gave wrong info",
    "the bot failed to update",
]

FAILURE_PHRASES = [
    "sorry, that price was incorrect",
    "apologies for the delayed response",
    "the information i gave earlier was wrong",
    "please wait, system is slow today",
]


def make_transcript(blame, rng):
    customer = rng.choice([
        "why is the price different from the page?",
        "i need warranty details before buying",
        "can you confirm delivery date?",
        "the link you sent does not work",
    ])
    agent = rng.choice(FAILURE_PHRASES)
    tail = ""
    if blame:
        tail = " " + rng.choice(BLAME_PHRASES) + "."
    return f"customer: {customer} | agent: {agent}{tail}"


def generate_sessions(n_sessions=5000, seed=42):
    rng = np.random.default_rng(seed)
    agents = rng.integers(1, 121, size=n_sessions)
    days = rng.integers(0, 92, size=n_sessions)
    day = pd.Timestamp("2024-05-01") + pd.to_timedelta(days, unit="D")

    peer_blame = rng.normal(0.04, 0.01, size=92)
    peer_delta = np.diff(peer_blame, prepend=peer_blame[0])

    blame_prob = 1 / (1 + np.exp(-(4.0 * peer_delta[days] + rng.normal(0, 0.25, n_sessions) - 2.2)))
    blame = rng.binomial(1, np.clip(blame_prob, 0.02, 0.20))

    vip = rng.integers(0, 5, size=n_sessions)
    prev_purchase = rng.poisson(1.5, size=n_sessions)
    is_fan = rng.binomial(1, 0.3, size=n_sessions)
    experience = rng.integers(0, 97, size=n_sessions)

    emotion = -0.15 + 0.18 * blame + 0.02 * vip + rng.normal(0, 0.45, n_sessions)
    engagement = 1.55 - 0.08 * blame + 0.03 * np.log1p(prev_purchase) + rng.normal(0, 0.35, n_sessions)
    purchase_logit = -2.6 - 0.35 * blame + 0.12 * np.log1p(prev_purchase) + 0.08 * is_fan + rng.normal(0, 0.4, n_sessions)
    purchase = (purchase_logit + rng.normal(0, 0.2, n_sessions) > 0).astype(int)

    return pd.DataFrame({
        "session_id": np.arange(n_sessions),
        "agent_id": agents,
        "day": day,
        "day_idx": days,
        "transcript": [make_transcript(int(b), rng) for b in blame],
        "service_failure": 1,
        "blame_true": blame,
        "peer_blame_delta": peer_delta[days],
        "customer_emotion": emotion,
        "log_engagement": engagement,
        "purchase": purchase,
        "vip_level": vip,
        "is_fan": is_fan,
        "previous_purchase": prev_purchase,
        "agent_experience": experience,
        "human_sentiment": rng.uniform(0.4, 0.9, n_sessions),
        "ai_sentiment": rng.uniform(0.3, 0.8, n_sessions),
        "ai_detect_ratio": rng.uniform(0, 1, n_sessions),
    })


df = generate_sessions()
df.head()

## 3. NLP Labeling Layer

This mirrors the paper's LLM-based detection step with a lightweight, auditable baseline:
- service failure detection
- blame detection
- customer sentiment scoring

You can replace these functions with Gemini or HuggingFace models in production.

In [ ]:
BLAME_PATTERNS = [r"system made a mistake", r"ai misunderstood", r"caused by the bot", r"chatbot", r"the bot", r"our ai"]
FAILURE_PATTERNS = [r"incorrect", r"wrong", r"delayed", r"slow", r"apolog", r"mistake"]


def detect_failure(text):
    t = text.lower()
    return int(any(re.search(p, t) for p in FAILURE_PATTERNS))


def detect_blame(text):
    t = text.lower()
    return int(any(re.search(p, t) for p in BLAME_PATTERNS))


def score_sentiment(text):
    neg = len(re.findall(r"angry|frustrated|upset|bad|terrible", text.lower()))
    pos = len(re.findall(r"thanks|great|good|helpful", text.lower()))
    return float(np.clip((pos - neg) / 10, -1, 1))


df["failure_detected"] = df["transcript"].map(detect_failure)
df["blame_detected"] = df["transcript"].map(detect_blame)

# Inject small labeling noise to mirror paper-level LLM agreement (~97.5%)
flip_rng = np.random.default_rng(42)
flip_n = int(0.025 * len(df))
flip_idx = flip_rng.choice(len(df), size=flip_n, replace=False)
df.loc[flip_idx, "blame_detected"] = 1 - df.loc[flip_idx, "blame_detected"].astype(int)

df["sentiment_score"] = df["transcript"].map(score_sentiment)

blame_auc = roc_auc_score(df["blame_true"], df["blame_detected"])
blame_acc = accuracy_score(df["blame_true"], df["blame_detected"])

pd.DataFrame({
    "metric": ["blame_accuracy", "blame_auc", "failure_detection_rate"],
    "value": [blame_acc, blame_auc, df["failure_detected"].mean()],
})

## 4. Two-Stage Control Function Estimation

Stage 1: blame ~ peer instrument + controls (logit)

Stage 2: outcomes ~ blame + control residual + controls

In [ ]:
controls = (
    "human_sentiment + ai_sentiment + ai_detect_ratio + vip_level + "
    "is_fan + previous_purchase + agent_experience"
)

first_stage = smf.logit(
    f"blame_detected ~ peer_blame_delta + {controls}",
    data=df,
).fit(disp=0)

df["control_residual"] = first_stage.resid_pearson

emotion_model = smf.ols(
    f"customer_emotion ~ blame_detected + control_residual + {controls}",
    data=df,
).fit(cov_type="HC1")

engagement_model = smf.ols(
    f"log_engagement ~ blame_detected + control_residual + {controls}",
    data=df,
).fit(cov_type="HC1")

purchase_model = smf.logit(
    f"purchase ~ blame_detected + control_residual + {controls}",
    data=df,
).fit(disp=0)

print(first_stage.summary().tables[1])
print("\nSecond-stage blame coefficients:")
for name, model in [
    ("emotion", emotion_model),
    ("engagement", engagement_model),
    ("purchase", purchase_model),
]:
    coef = model.params["blame_detected"]
    pval = model.pvalues["blame_detected"]
    print(f"{name:12s} coef={coef: .4f} p={pval:.4g}")

## 5. Results Visualization

In [ ]:
results = pd.DataFrame({
    "outcome": ["Emotion", "Engagement", "Purchase"],
    "coef": [
        emotion_model.params["blame_detected"],
        engagement_model.params["blame_detected"],
        purchase_model.params["blame_detected"],
    ],
})

results["effect_pct"] = np.where(
    results["outcome"].eq("Emotion"),
    results["coef"],
    (np.exp(results["coef"]) - 1) * 100,
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

sns.barplot(data=results, x="outcome", y="coef", ax=axes[0], palette="Blues_d")
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_title("Blame Effect Coefficients")
axes[0].set_xlabel("Outcome")
axes[0].set_ylabel("Estimated coefficient")

plot_df = df.groupby("blame_detected")[["customer_emotion", "log_engagement", "purchase"]].mean().reset_index()
plot_df["blame_detected"] = plot_df["blame_detected"].map({0: "No Blame", 1: "Blame"})

plot_df.melt("blame_detected", var_name="metric", value_name="value").pipe(
    lambda d: sns.barplot(data=d, x="metric", y="value", hue="blame_detected", ax=axes[1])
)
axes[1].set_title("Mean Outcomes by Blame Status")
axes[1].set_xlabel("Metric")

plt.tight_layout()
plt.show()

results

## 6. Relationship Strength Moderator (ERA Mechanism Check)

In [ ]:
moderation_emotion = smf.ols(
    f"customer_emotion ~ blame_detected * previous_purchase + control_residual + {controls}",
    data=df,
).fit(cov_type="HC1")

moderation_engagement = smf.ols(
    f"log_engagement ~ blame_detected * previous_purchase + control_residual + {controls}",
    data=df,
).fit(cov_type="HC1")

moderation_purchase = smf.logit(
    f"purchase ~ blame_detected * previous_purchase + control_residual + {controls}",
    data=df,
).fit(disp=0)

interaction_table = pd.DataFrame({
    "model": ["emotion", "engagement", "purchase"],
    "interaction_coef": [
        moderation_emotion.params["blame_detected:previous_purchase"],
        moderation_engagement.params["blame_detected:previous_purchase"],
        moderation_purchase.params["blame_detected:previous_purchase"],
    ],
    "interaction_p": [
        moderation_emotion.pvalues["blame_detected:previous_purchase"],
        moderation_engagement.pvalues["blame_detected:previous_purchase"],
        moderation_purchase.pvalues["blame_detected:previous_purchase"],
    ],
})

interaction_table

## 7. Policy Summary (ERA Lens)

**Short-term emotion:** blaming AI can reduce anger toward the human agent.

**Behavioral cost:** the same blame signal implies poor human-AI coordination, reducing engagement and purchase conversion.

**Managerial implication:** train agents to repair failures without scapegoating AI, and monitor coordination cues visible to customers.

### Suggested email sentence to professor

"I prepared a reproducible Colab pipeline extending your hybrid-service blame framework with modern Python NLP and econometric modules, including IV-based endogeneity correction and ERA-aligned moderation checks."

## Pipeline Workflow

The full process is shown below (also saved as `assets/workflow_architecture_cartoon.png`).

**Simple pipeline in plain language:**
1. Collect hybrid-service chat sessions after failures.
2. Label blame and customer sentiment from text.
3. Build three outcomes: emotion, engagement, purchase.
4. Estimate blame effects with IV + control-function correction.
5. Test ERA moderation and translate results into service policy.

In [ ]:
from IPython.display import Image, display

display(Image("assets/workflow_architecture_cartoon.png"))